In [1]:
import torch
from tokenizers import Tokenizer


train_tokens = torch.load("train_tokens_6.pt")
val_tokens   = torch.load("val_tokens_6.pt")
test_tokens  = torch.load("test_tokens_6.pt")

tokenizer = Tokenizer.from_file("svg_tokenizer_6.json")

print(len(train_tokens), len(val_tokens), len(test_tokens))

199332 2034 2035


In [2]:
def count_tokens(data):
    return sum(len(x) for x in data)

print("Train tokens:", count_tokens(train_tokens))
print("Val tokens:", count_tokens(val_tokens))
print("Test tokens:", count_tokens(test_tokens))

Train tokens: 120374653
Val tokens: 1233454
Test tokens: 1217426


In [2]:
# -*- coding: utf-8 -*-
"""Part 2: Transformer Scaling Study for SVG token modeling.

This script performs a learning rate sweep on the smallest model and then trains
five decoder-only transformer sizes for exactly one epoch each.

It logs validation loss, training loss curves, wall-clock time, throughput, and
(optional) GPU memory peak usage. It also fits a scaling law of the form
  L = a * N^{-alpha} + c
where N is the number of model parameters.
"""

import math
import os
import random
import time

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tokenizers import Tokenizer




TOKENIZER_PATH = os.path.join('svg_tokenizer_6.json')
TRAIN_TOKENS_PATH = os.path.join( 'train_tokens_6.pt')
VAL_TOKENS_PATH = os.path.join('val_tokens_6.pt')
TEST_TOKENS_PATH = os.path.join('test_tokens_6.pt')
OUTPUT_DIR = os.path.join( 'part2_scaling_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

for path in [TOKENIZER_PATH, TRAIN_TOKENS_PATH, VAL_TOKENS_PATH, TEST_TOKENS_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing required file: {path}')


def safe_token_id(tokenizer, token):
    try:
        return tokenizer.token_to_id(token)
    except Exception:
        return None


def count_tokens(sequences):
    return sum(len(seq) for seq in sequences)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=1024):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + self.pe[:seq_len, :].unsqueeze(0)
        return self.dropout(x)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        assert self.head_dim * n_heads == d_model, 'd_model must be divisible by n_heads'
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.q_linear(query)
        K = self.k_linear(key)
        V = self.v_linear(value)
        Q = Q.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        energy = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float('-inf'))
        energy = torch.clamp(energy, min=-50.0, max=50.0)
        attention = torch.softmax(energy, dim=-1)
        x = torch.matmul(self.dropout(attention), V)
        x = x.permute(0, 2, 1, 3).contiguous()
        x = x.view(batch_size, -1, self.d_model)
        return self.fc_out(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x


class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1, max_len=1024, padding_idx=None):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.positional_encoding = PositionalEncoding(d_model, dropout, max_len)
        self.initial_layer_norm = nn.LayerNorm(d_model)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def _generate_square_subsequent_mask(self, sz, device):
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1).bool()
        return ~mask

    def forward(self, src):
        src_len = src.size(1)
        mask = self._generate_square_subsequent_mask(src_len, src.device)
        src = self.token_embedding(src)
        src = self.positional_encoding(src * math.sqrt(self.token_embedding.embedding_dim))
        src = self.initial_layer_norm(src)
        for layer in self.layers:
            src = layer(src, mask)
        src = self.norm(src)
        return self.fc_out(src)


class ModelConfig:
    def __init__(self, name, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        self.name = name
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.d_ff = d_ff
        self.dropout = dropout


class TokenDataset:
    def __init__(self, sequences):
        self.data = sequences

    def sample_batch(self, batch_size, block_size, device):
        x_batch = []
        y_batch = []
        while len(x_batch) < batch_size:
            seq = random.choice(self.data)
            if len(seq) < block_size + 1:
                continue
            start = random.randint(0, len(seq) - block_size - 1)
            chunk = seq[start:start + block_size + 1]
            x_batch.append(torch.tensor(chunk[:-1], dtype=torch.long))
            y_batch.append(torch.tensor(chunk[1:], dtype=torch.long))
        x = torch.stack(x_batch).to(device)
        y = torch.stack(y_batch).to(device)
        return x, y


def get_scheduler(optimizer, total_steps, warmup_steps):
    warmup_scheduler = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=warmup_steps)
    cosine_scheduler = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
    return SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_steps])


def steps_per_epoch(total_tokens, batch_size, block_size):
    tokens_per_step = batch_size * block_size
    return max(1, math.ceil(total_tokens / tokens_per_step))


def train_one_epoch(model, dataset, optimizer, scheduler, device, steps, batch_size, block_size, log_interval=100):
    model.train()
    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    losses = []
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    for step in range(1, steps + 1):
        x, y = dataset.sample_batch(batch_size, block_size, device)
        outputs = model(x)
        loss = criterion(outputs.view(-1, outputs.size(-1)), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        if step % log_interval == 0 or step == steps:
            interval_count = min(log_interval, step)
            avg_loss = running_loss / interval_count
            losses.append(avg_loss)
            print(f'Step {step}/{steps}, train_loss={avg_loss:.4f}, lr={scheduler.get_last_lr()[0]:.6g}')
            running_loss = 0.0

    elapsed = time.time() - start_time
    throughput = steps * batch_size * block_size / elapsed
    peak_memory_gb = None
    if device == 'cuda':
        peak_memory_gb = torch.cuda.max_memory_allocated() / 1024**3
    return losses, elapsed, throughput, peak_memory_gb


def evaluate(model, dataset, device, steps=200, batch_size=8, block_size=512):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    with torch.no_grad():
        for _ in range(steps):
            x, y = dataset.sample_batch(batch_size, block_size, device)
            outputs = model(x)
            loss = criterion(outputs.view(-1, outputs.size(-1)), y.view(-1))
            total_loss += loss.item()
    return total_loss / steps


def fit_scaling_law(params, losses):
    params = np.asarray(params)
    losses = np.asarray(losses)
    best = None
    for c in np.linspace(0.0, losses.min() * 0.9, 200):
        adjusted = losses - c
        if np.any(adjusted <= 0):
            continue
        log_params = np.log(params)
        log_adjusted = np.log(adjusted)
        coeffs = np.polyfit(log_params, log_adjusted, 1)
        alpha = -coeffs[0]
        a = math.exp(coeffs[1])
        prediction = a * params ** (-alpha) + c
        rss = np.sum((prediction - losses) ** 2)
        if best is None or rss < best['rss']:
            best = {'a': a, 'alpha': alpha, 'c': c, 'rss': rss}
    if best is None:
        raise RuntimeError('Scaling law fit failed; check loss values.')
    return best


def plot_scaling_curve(results, fit_result, output_dir):
    params = [r['params'] for r in results]
    losses = [r['val_loss'] for r in results]
    plt.figure(figsize=(8, 6))
    plt.plot(params, losses, 'o', label='Measured')
    x_line = np.logspace(math.log10(min(params)), math.log10(max(params)), 100)
    y_line = fit_result['a'] * x_line ** (-fit_result['alpha']) + fit_result['c']
    plt.plot(x_line, y_line, '-', label=f'Fit: a*N^-{fit_result["alpha"]:.3f}+c')
    plt.xscale('log')
    plt.xlabel('Number of Parameters (M)')
    plt.ylabel('Validation Loss')
    plt.title('Transformer Scaling Law')
    plt.grid(True, which='both', ls='--', alpha=0.5)
    plt.legend()
    path = os.path.join(output_dir, 'scaling_plot.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    return path


def plot_training_curves(curves, output_dir):
    plt.figure(figsize=(10, 6))
    for name, values in curves.items():
        plt.plot(values, label=name)
    plt.xlabel('Log Interval')
    plt.ylabel('Training Loss')
    plt.title('Training Loss Curves')
    plt.legend()
    plt.grid(True)
    path = os.path.join(output_dir, 'training_curves.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    return path

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
train_tokens = torch.load(TRAIN_TOKENS_PATH)
val_tokens = torch.load(VAL_TOKENS_PATH)
test_tokens = torch.load(TEST_TOKENS_PATH)

total_train_tokens = count_tokens(train_tokens)
total_val_tokens = count_tokens(val_tokens)
total_test_tokens = count_tokens(test_tokens)

print('Data summary:')
print(f'  train tokens: {total_train_tokens}')
print(f'  val tokens:   {total_val_tokens}')
print(f'  test tokens:  {total_test_tokens}')
if total_train_tokens < 100_000_000:
    print('WARNING: training data contains fewer than 100M tokens; using available tokens for one epoch.')

vocab_size = tokenizer.get_vocab_size()
pad_id = safe_token_id(tokenizer, '[PAD]')
if pad_id is None:
    pad_id = safe_token_id(tokenizer, '<pad>')
if pad_id is None:
    print('No explicit pad token found; training uses fixed-length crops without padding_idx.')

print(f'Vocabulary size: {vocab_size}')
print(f'Device: {device}')

train_dataset = TokenDataset(train_tokens)
val_dataset = TokenDataset(val_tokens)

Data summary:
  train tokens: 120374653
  val tokens:   1233454
  test tokens:  1217426
Vocabulary size: 6000
Device: cuda


In [4]:
block_size = 512
tokens_per_batch = 4096
batch_size = max(32, tokens_per_batch // block_size)
epoch_steps = steps_per_epoch(total_train_tokens, batch_size, block_size)
warmup_steps = max(1, epoch_steps // 5)

print(f'Block size: {block_size}, batch_size: {batch_size}, epoch_steps: {epoch_steps}, warmup_steps: {warmup_steps}')


Block size: 512, batch_size: 32, epoch_steps: 7348, warmup_steps: 1469


In [4]:
lr_candidates = np.logspace(-5, -3, 7)
best_lr = None
best_val_loss = float('inf')
lr_sweep_results = []

print('\n===== Learning Rate Sweep =====')
tiny_cfg = ModelConfig('tiny', 128, 4, 4, 512)
for lr in lr_candidates:
    model = DecoderOnlyTransformer(vocab_size=vocab_size, d_model=tiny_cfg.d_model, n_layers=tiny_cfg.n_layers, n_heads=tiny_cfg.n_heads, d_ff=tiny_cfg.d_ff, max_len=block_size, padding_idx=None).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_scheduler(optimizer, total_steps=epoch_steps, warmup_steps=warmup_steps)
    train_one_epoch(model, train_dataset, optimizer, scheduler, device, steps=min(epoch_steps, 800), batch_size=batch_size, block_size=block_size, log_interval=200)
    val_loss = evaluate(model, val_dataset, device, steps=100, batch_size=batch_size, block_size=block_size)
    lr_sweep_results.append({'lr': lr, 'val_loss': val_loss})
    print(f'lr={lr:.2e} | val_loss={val_loss:.4f}')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_lr = lr

print(f'Best LR = {best_lr:.2e} with val_loss = {best_val_loss:.4f}')
with open(os.path.join(OUTPUT_DIR, 'lr_sweep.csv'), 'w') as f:
    f.write('lr,val_loss\n')
    for row in lr_sweep_results:
        f.write(f"{row['lr']:.6e},{row['val_loss']:.6f}\n")


===== Learning Rate Sweep =====
Step 200/800, train_loss=8.8026, lr=2.7248e-06
Step 400/800, train_loss=8.2515, lr=5.4496e-06
Step 600/800, train_loss=7.1605, lr=8.17439e-06


/home/av4008/.local/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:198: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Step 800/800, train_loss=6.2164, lr=9.98754e-06
lr=1.00e-05 | val_loss=5.7473
Step 200/800, train_loss=8.5190, lr=5.87041e-06
Step 400/800, train_loss=7.3111, lr=1.17408e-05
Step 600/800, train_loss=5.8941, lr=1.76112e-05
Step 800/800, train_loss=4.7042, lr=2.15175e-05
lr=2.15e-05 | val_loss=3.9662
Step 200/800, train_loss=8.4567, lr=1.26474e-05
Step 400/800, train_loss=6.4471, lr=2.52948e-05
Step 600/800, train_loss=4.6165, lr=3.79422e-05
Step 800/800, train_loss=2.9213, lr=4.6358e-05
lr=4.64e-05 | val_loss=2.3314
Step 200/800, train_loss=7.9702, lr=2.7248e-05
Step 400/800, train_loss=5.2363, lr=5.4496e-05
Step 600/800, train_loss=2.7750, lr=8.17439e-05
Step 800/800, train_loss=1.9971, lr=9.98754e-05
lr=1.00e-04 | val_loss=1.8721
Step 200/800, train_loss=7.2762, lr=5.87041e-05
Step 400/800, train_loss=3.3899, lr=0.000117408
Step 600/800, train_loss=1.9551, lr=0.000176112
Step 800/800, train_loss=1.7750, lr=0.000215175
lr=2.15e-04 | val_loss=1.6756
Step 200/800, train_loss=6.5586, lr=0

In [5]:
# Save best LR for later use
with open(os.path.join(OUTPUT_DIR, "best_lr.txt"), "w") as f:
    f.write(str(best_lr))

In [5]:
def run_model(cfg, best_lr):
    print(f'\n===== Training {cfg.name.upper()} =====')

    model = DecoderOnlyTransformer(
        vocab_size=vocab_size,
        d_model=cfg.d_model,
        n_layers=cfg.n_layers,
        n_heads=cfg.n_heads,
        d_ff=cfg.d_ff,
        max_len=block_size,
        padding_idx=None
    ).to(device)

    param_count = sum(p.numel() for p in model.parameters()) / 1e6

    optimizer = optim.AdamW(model.parameters(), lr=best_lr)
    scheduler = get_scheduler(optimizer, total_steps=epoch_steps, warmup_steps=warmup_steps)

    train_losses, epoch_time, throughput, peak_memory = train_one_epoch(
        model, train_dataset, optimizer, scheduler, device,
        steps=epoch_steps, batch_size=batch_size, block_size=block_size
    )

    val_loss = evaluate(
        model, val_dataset, device,
        steps=100, batch_size=batch_size, block_size=block_size
    )

    result = {
        'name': cfg.name,
        'params': param_count,
        'val_loss': val_loss,
        'train_loss_last': train_losses[-1] if train_losses else float('nan'),
        'epoch_time': epoch_time,
        'throughput': throughput,
        'peak_memory_gb': peak_memory,
        'train_curve': train_losses,
    }

    print(f"{cfg.name:>6} | {param_count:.2f}M params | val_loss={val_loss:.4f}")

    # ✅ Save immediately (important for crash recovery)
    torch.save(result, os.path.join(OUTPUT_DIR, f"{cfg.name}.pt"))

    return result

In [6]:
MODEL_CONFIGS = {
    'tiny':   ModelConfig('tiny', 128, 4, 4, 512),
    'small':  ModelConfig('small', 192, 6, 6, 768),
    'medium': ModelConfig('medium', 384, 6, 6, 1536),
    'large':  ModelConfig('large', 512, 8, 8, 2048),
    'xl':     ModelConfig('xl', 768, 12, 12, 3072),
}

In [7]:
def main(model_name):
    assert model_name in MODEL_CONFIGS, f"Invalid model: {model_name}"

    # load best LR
    with open(os.path.join(OUTPUT_DIR, "best_lr.txt")) as f:
        best_lr = float(f.read().strip())

    cfg = MODEL_CONFIGS[model_name]

    run_model(cfg, best_lr)

In [15]:
main('tiny')



===== Training TINY =====
Step 100/7348, train_loss=7.8219, lr=6.80745e-05
Step 200/7348, train_loss=4.8304, lr=0.000136148
Step 300/7348, train_loss=2.4382, lr=0.000204221
Step 400/7348, train_loss=1.8974, lr=0.000272295
Step 500/7348, train_loss=1.7888, lr=0.000340368
Step 600/7348, train_loss=1.7195, lr=0.000408442
Step 700/7348, train_loss=1.6560, lr=0.000476515
Step 800/7348, train_loss=1.6145, lr=0.000544589
Step 900/7348, train_loss=1.5635, lr=0.000612662
Step 1000/7348, train_loss=1.5222, lr=0.000680736
Step 1100/7348, train_loss=1.4852, lr=0.000748809
Step 1200/7348, train_loss=1.4378, lr=0.000816882
Step 1300/7348, train_loss=1.3836, lr=0.000884956
Step 1400/7348, train_loss=1.3398, lr=0.000953029
Step 1500/7348, train_loss=1.3169, lr=0.000999931
Step 1600/7348, train_loss=1.2721, lr=0.000998775
Step 1700/7348, train_loss=1.2379, lr=0.000996195
Step 1800/7348, train_loss=1.2084, lr=0.000992199
Step 1900/7348, train_loss=1.1753, lr=0.000986797
Step 2000/7348, train_loss=1.148

In [16]:
main('small')



===== Training SMALL =====
Step 100/7348, train_loss=7.0201, lr=6.80745e-05
Step 200/7348, train_loss=3.3733, lr=0.000136148
Step 300/7348, train_loss=1.9452, lr=0.000204221
Step 400/7348, train_loss=1.7553, lr=0.000272295
Step 500/7348, train_loss=1.6712, lr=0.000340368
Step 600/7348, train_loss=1.5989, lr=0.000408442
Step 700/7348, train_loss=1.5315, lr=0.000476515
Step 800/7348, train_loss=1.4386, lr=0.000544589
Step 900/7348, train_loss=1.3350, lr=0.000612662
Step 1000/7348, train_loss=1.2949, lr=0.000680736
Step 1100/7348, train_loss=1.2223, lr=0.000748809
Step 1200/7348, train_loss=1.1647, lr=0.000816882
Step 1300/7348, train_loss=1.1161, lr=0.000884956
Step 1400/7348, train_loss=1.0623, lr=0.000953029
Step 1500/7348, train_loss=1.0310, lr=0.000999931
Step 1600/7348, train_loss=0.9909, lr=0.000998775
Step 1700/7348, train_loss=0.9646, lr=0.000996195
Step 1800/7348, train_loss=0.9478, lr=0.000992199
Step 1900/7348, train_loss=0.9205, lr=0.000986797
Step 2000/7348, train_loss=0.91

In [17]:
main('medium')



===== Training MEDIUM =====
Step 100/7348, train_loss=5.6603, lr=6.80745e-05
Step 200/7348, train_loss=2.0771, lr=0.000136148
Step 300/7348, train_loss=1.7239, lr=0.000204221
Step 400/7348, train_loss=1.6044, lr=0.000272295
Step 500/7348, train_loss=1.4621, lr=0.000340368
Step 600/7348, train_loss=1.3416, lr=0.000408442
Step 700/7348, train_loss=1.2703, lr=0.000476515
Step 800/7348, train_loss=1.2049, lr=0.000544589
Step 900/7348, train_loss=1.1295, lr=0.000612662
Step 1000/7348, train_loss=1.0479, lr=0.000680736
Step 1100/7348, train_loss=1.0185, lr=0.000748809
Step 1200/7348, train_loss=0.9753, lr=0.000816882
Step 1300/7348, train_loss=0.9358, lr=0.000884956
Step 1400/7348, train_loss=0.9157, lr=0.000953029
Step 1500/7348, train_loss=0.9024, lr=0.000999931
Step 1600/7348, train_loss=0.8837, lr=0.000998775
Step 1700/7348, train_loss=0.8516, lr=0.000996195
Step 1800/7348, train_loss=0.8340, lr=0.000992199
Step 1900/7348, train_loss=0.8325, lr=0.000986797
Step 2000/7348, train_loss=0.8

In [18]:
main('large')



===== Training LARGE =====
Step 100/7348, train_loss=4.8602, lr=6.80745e-05
Step 200/7348, train_loss=1.8699, lr=0.000136148
Step 300/7348, train_loss=1.6427, lr=0.000204221
Step 400/7348, train_loss=1.4815, lr=0.000272295
Step 500/7348, train_loss=1.3193, lr=0.000340368
Step 600/7348, train_loss=1.1952, lr=0.000408442
Step 700/7348, train_loss=1.1059, lr=0.000476515
Step 800/7348, train_loss=1.0166, lr=0.000544589
Step 900/7348, train_loss=0.9913, lr=0.000612662
Step 1000/7348, train_loss=0.9293, lr=0.000680736
Step 1100/7348, train_loss=0.8722, lr=0.000748809
Step 1200/7348, train_loss=0.8504, lr=0.000816882
Step 1300/7348, train_loss=0.9046, lr=0.000884956
Step 1400/7348, train_loss=0.8172, lr=0.000953029
Step 1500/7348, train_loss=0.8042, lr=0.000999931
Step 1600/7348, train_loss=0.9720, lr=0.000998775
Step 1700/7348, train_loss=0.9514, lr=0.000996195
Step 1800/7348, train_loss=0.7824, lr=0.000992199
Step 1900/7348, train_loss=0.7559, lr=0.000986797
Step 2000/7348, train_loss=0.73

In [ ]:
main('xl')


===== Training XL =====
Step 100/7348, train_loss=3.8764, lr=6.80745e-05
Step 200/7348, train_loss=1.7248, lr=0.000136148
Step 300/7348, train_loss=1.5094, lr=0.000204221
Step 400/7348, train_loss=1.2992, lr=0.000272295
Step 500/7348, train_loss=1.1685, lr=0.000340368
Step 600/7348, train_loss=1.0319, lr=0.000408442
Step 700/7348, train_loss=0.9395, lr=0.000476515
Step 800/7348, train_loss=1.2856, lr=0.000544589
Step 900/7348, train_loss=1.1678, lr=0.000612662
Step 1000/7348, train_loss=0.8753, lr=0.000680736
Step 1100/7348, train_loss=0.8359, lr=0.000748809
Step 1200/7348, train_loss=0.8136, lr=0.000816882
Step 1300/7348, train_loss=0.8117, lr=0.000884956
Step 1400/7348, train_loss=3.3717, lr=0.000953029


/home/av4008/.local/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:198: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Step 1500/7348, train_loss=3.5417, lr=0.000999931
Step 1600/7348, train_loss=3.5340, lr=0.000998775
Step 1700/7348, train_loss=3.5313, lr=0.000996195
Step 1800/7348, train_loss=3.5345, lr=0.000992199
Step 1900/7348, train_loss=3.5279, lr=0.000986797
Step 2000/7348, train_loss=3.5247, lr=0.000980006
Step 2100/7348, train_loss=3.5254, lr=0.000971844
Step 2200/7348, train_loss=3.5358, lr=0.000962335
Step 2300/7348, train_loss=3.5209, lr=0.000951506
Step 2400/7348, train_loss=3.5318, lr=0.000939388
Step 2500/7348, train_loss=3.5350, lr=0.000926016
Step 2600/7348, train_loss=3.5194, lr=0.000911428
Step 2700/7348, train_loss=3.5268, lr=0.000895665
Step 2800/7348, train_loss=3.5264, lr=0.000878772
Step 2900/7348, train_loss=3.5302, lr=0.000860798
Step 3000/7348, train_loss=3.5217, lr=0.000841794
Step 3100/7348, train_loss=3.5227, lr=0.000821814
Step 3200/7348, train_loss=3.5219, lr=0.000800916
Step 3300/7348, train_loss=3.5300, lr=0.000779158
Step 3400/7348, train_loss=3.5271, lr=0.000756604


In [ ]:
def run_scaling_analysis():
    results = []

    for name in MODEL_CONFIGS.keys():
        path = os.path.join(OUTPUT_DIR, f"{name}.pt")
        if os.path.exists(path):
            results.append(torch.load(path))

    if len(results) < 2:
        print("Not enough models trained for scaling law.")
        return

    fit_result = fit_scaling_law(
        [r['params'] for r in results],
        [r['val_loss'] for r in results]
    )

    print('\nScaling fit:')
    print(f"  a = {fit_result['a']:.6f}")
    print(f"  alpha = {fit_result['alpha']:.6f}")
    print(f"  c = {fit_result['c']:.6f}")

    plot_scaling_curve(results, fit_result, OUTPUT_DIR)

    training_curves = {r['name']: r['train_curve'] for r in results}
    plot_training_curves(training_curves, OUTPUT_DIR)
run_scaling_analysis()

In [7]:
print("Vocab size:", vocab_size)

all_tokens = torch.cat(train_tokens) if isinstance(train_tokens, list) else train_tokens

print("Min token:", all_tokens.min().item())
print("Max token:", all_tokens.max().item())

Vocab size: 4000
Min token: 4
Max token: 5999
